<a href="https://colab.research.google.com/github/vulpecriptografica/Symm4ML/blob/main/rep2_companion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group Representations II — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Group Representations II exercise section by section. Use it to **prototype your code** and **test your implementations** against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/rep2_companion.ipynb)

## Setup

In [1]:
%%capture
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [2]:
import itertools
from typing import List, Set, FrozenSet

import numpy as np

from symm4ml import groups, linalg, rep

### Reference data

These matrices and tables are used throughout the exercise for testing.

In [3]:
# P(3) multiplication table
p3_table = np.array([
    [0, 1, 2, 3, 4, 5],
    [1, 0, 4, 5, 2, 3],
    [2, 5, 0, 4, 3, 1],
    [3, 4, 5, 0, 1, 2],
    [4, 3, 1, 2, 5, 0],
    [5, 2, 3, 1, 0, 4],
])

# P(3) irreps for testing
p3_irrep_trivial = np.array([[[1.0]], [[1.0]], [[1.0]], [[1.0]], [[1.0]], [[1.0]]])

p3_irrep_sign = np.array([[[1.0]], [[-1.0]], [[-1.0]], [[-1.0]], [[1.0]], [[1.0]]])

p3_irrep_rot = np.array([
    [[1.0, 0.0], [0.0, 1.0]],
    [[-0.2916587, 0.95652245], [0.95652245, 0.2916587]],
    [[-0.6825434, -0.73084507], [-0.73084507, 0.6825434]],
    [[0.97420209, -0.22567739], [-0.22567739, -0.97420209]],
    [[-0.5, 0.8660254], [-0.8660254, -0.5]],
    [[-0.5, -0.8660254], [0.8660254, -0.5]],
])

# D2 (Klein four-group) table
ans_table2 = np.array([[0, 1, 2, 3], [1, 0, 3, 2], [2, 3, 0, 1], [3, 2, 1, 0]])

# Z2 table
z2_table = np.array([[0, 1], [1, 0]])

---
## 1. `similarity_transform(rep, U)`

Compute the similarity transform $\rho'(g) = U \rho(g) U^{-1}$ of a representation. $U$ is an invertible (not necessarily unitary) complex matrix.

In [4]:
def similarity_transform(rep: np.array, U: np.array) -> np.array:
    """Returns transformed representation U rep U^{-1}.
    Input:
        rep: np.array [n, d, d] representation of the group. rep[i] is a matrix that
            represents the representation at the i-th element of group.
        U: np.array [d, d] invertible complex matrix
    Output:
        Transformed representation. np.array [n, d, d]
    """
    pass  # your code here

In [5]:
# No small tests for this function

---
## 2. `character_table(irreps, conj_classes)`

Compute the character table for a group given its irreps and conjugacy classes. The character of an irrep for a given class is the trace of its matrix representation for any element in that class.

In [6]:
def character_table(irreps: List[np.array], conj_classes: Set[FrozenSet[int]]) -> np.ndarray:
    """Returns character table for a group.
    Input:
        irreps: List of np.arrays of shape [n, d, d], where n is the order of the group and d is the dimension of the irrep (which may vary).
        conj_classes: List of sets of integers, where the total number of integers across all sets in the list is n.
                      Each set contains elements of a conjugacy class.
    Output:
        Character table. np.array [len(irreps), len(conj_classes)] (where the class / irrep order us unchanged)
    """
    pass  # your code here

In [7]:
# No small tests for this function

---
## 3. `regular_representation(table)`

Compute the left regular representation from a group's multiplication table. For a group with $h$ elements, $D(g)|h\rangle = |gh\rangle$.

See the [Wikipedia page](https://en.wikipedia.org/wiki/Regular_representation) for reference.

In [8]:
def regular_representation(table: np.array) -> np.array:
    """Returns regular representation for group represented by a multiplication table.
    Input:
        table: np.array [n, n] where table[i, j] = k means i * j = k.
    Output:
        Regular representation. array [n, n, n] where reg_rep[i, :, :] = D(i) and D(i)e_j = e_{ij}.
                                Equivalently, D(g) |h> = |gh>
    """
    pass  # your code here

In [9]:
assert np.allclose(
    regular_representation(np.array([[0, 1], [1, 0]])),
    np.array([[[1.0, 0.0], [0.0, 1.0]], [[0.0, 1.0], [1.0, 0.0]]]),
)

TypeError: unsupported operand type(s) for -: 'NoneType' and 'float'

---
## 4. `decompose_rep_into_irreps(rep)`

Decompose a reducible representation into its irreducible components. Given $\rho = U(\rho_1 \oplus \dots \oplus \rho_r)U^\dagger$, recover $\{\rho_1, \dots, \rho_r\}$ up to isomorphism.

**Algorithm sketch:**
1. Find the space of matrices $Q$ satisfying $Q\rho = \rho Q$ using `linalg.infer_change_of_basis(rep, rep)`.
2. Take a random linear combination $\bar{Q} = \sum_i \alpha_i Q_i$ to break accidental degeneracies.
3. Compute the eigenspaces of $\bar{Q}$ — each eigenspace corresponds to an irrep.
4. For each eigenspace with orthonormal basis $B_i$, extract the irrep as $\rho_i = B_i^\dagger \rho B_i$.

Functions you may need: `linalg.infer_change_of_basis`, `np.random.rand`, `np.linalg.eig` or `np.linalg.eigh`, `linalg.eigenspaces`, and `linalg.gram_schmidt`.

<details>
<summary>Code for <code>linalg.eigenspaces</code> and <code>linalg.unique_with_tol</code></summary>

```python
def unique_with_tol(a, *, tol):
    """Find unique elements of an array with a tolerance."""
    assert a.ndim >= 1
    shape = a.shape
    a = a.reshape(len(a), -1)
    distances = np.linalg.norm(a[:, None] - a[None, :], axis=-1)
    inverses = -1 * np.ones(len(a), dtype=int)
    index = 0
    while True:
        (m,) = np.nonzero(inverses == -1)
        if len(m) == 0:
            break
        i = m[0]
        if np.any(inverses[distances[i] < tol] != -1):
            raise ValueError("The clusters are not clearly distinct.")
        inverses[distances[i] < tol] = index
        index += 1
    centers = np.zeros((np.max(inverses) + 1, a.shape[1]), dtype=a.dtype)
    np.add.at(centers, inverses, a)
    centers /= np.bincount(inverses)[:, None]
    centers = centers.reshape(len(centers), *shape[1:])
    return centers, inverses

def eigenspaces(val, vec, *, tol=1e-8):
    """Regroup eigenvectors by eigenvalues."""
    unique_val, i = unique_with_tol(val, tol=tol)
    return [(v, vec[:, i == j]) for j, v in enumerate(unique_val)]
```
</details>

In [44]:
def decompose_rep_into_irreps(rep: np.array, *, tol: float=1e-08) -> List[np.array]:
    """Decomposes representation into irreducible representations.
    Input:
        rep: np.array [n, d, d] representation of group. rep[g] is a matrix that
            represents g-th element of group.
    Output:
        Irreducible representations. List of np.array [n, d_i, d_i] where d_i is a dimension of i-th irrep.
                                     Note: you can output the irreps in any order and in any basis.
                                     If an irrep is included multiple times in the decomposition (i.e. with multiplicity greater than 1), please simply include it multiple times in the output list.
    """
    # step 1: find change of basis Q
    Qs = linalg.infer_change_of_basis(rep, rep)

    # step 2: random lin comb to break accidental degeneracies
    # this helps us get different eigenspaces -> diff similarity transforms B
    d = len(Qs)
    a = np.random.rand(d)
    Q_ = np.einsum('i,ijk->jk', a, Qs)

    # step 3: find eigenspaces of Q_ (one-to-one w/ irreps)
    evals, evecs = np.linalg.eig(Q_)
    espaces = linalg.eigenspaces(evals, evecs) # d x n for evec matrices

    # step 4: orthonormalize and extract irreps
    irreps = []
    l = len(espaces)
    for i in range(l): # for each espace
      B = espaces[i][1] # arr of evecs
      irr = np.einsum('ia, hij, jb -> hab' , B.conj(), rep, B)
      irreps.append(irr)
    return irreps

# (h,d,d) -> (h, n, n) via projection
# h = group dim,
# n indexed by a or b
# d indexed by i or j

# h = group dim
# n = dim of eigenspace/irrep, indexed by a or b
# d = dim of reducible (og) rep, indexed by i or j


In [ ]:
# No small tests for this function


      # P = B_dag @ B
      # B = linalg.gram_schmidt(now)[0] # mat of obas vecs
      # print('this is B:', B)
      # B_dag = B.conj().T
      # print(rep[i].shape)
      # B = espaces[i][1]
      # B_dag = B.conj().T
      # print(B.shape)

---
## 5. `infer_irreps(table)`

Generate the unique irreps of a group from its multiplication table. Start by computing the regular representation, decompose it, then remove duplicates using `are_isomorphic`.

In [29]:
def infer_irreps(table: np.array, *, tol: float=1e-08) -> List[np.array]:
    """Infers irreducible representations of group represented by multiplication table.
    Input:
        table: np.array [n, n] where table[i, j] = k means i * j = k.
    Output:
        Irreducible representations. List of np.array [n, d, d] where d is a dimension of irrep.
                                     Note: you can output the irreps in any order and in any basis.
    """
    reg_rep = rep.regular_representation(table)
    irreps = rep.decompose_rep_into_irreps(reg_rep) # list of irreps w/ duplicates
    idxs = []
    for i in range(len(irreps)-1):
        if any(rep.are_isomorphic(arr,irreps[i]) for arr in irreps[i+1:]):
          idxs.append(i)
    print('these are the irreps:', len(irreps))
    print('indices to delete:', idxs)
    unique = np.delete(irreps, idxs, axis=1)
    return unique


# ans = [rep.is_an_irrep(np.array([[0, 1], [1, 0]]), rho) for rho in infer_irreps(np.array([[0, 1], [1, 0]]))]
# print(ans)
# # [True, True]

# ans = [rep.is_an_irrep(ans_table2, rho) for rho in infer_irreps(ans_table2)]
# print(ans)
# # [True, True, True, True]

ans = [is_an_irrep(p3_table, rho) for rho in infer_irreps(p3_table)]
print(ans)

these are the irreps: 4
indices to delete: [2]


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (4, 6) + inhomogeneous part.

In [ ]:
# No small tests for this function

---
## 6. `tensor_product(rep1, rep2)`

Compute the tensor product of two representations: $\rho(g) = \rho_1(g) \otimes \rho_2(g)$.

You can either use `np.einsum` followed by a `reshape`, or use `np.kron`.

In [ ]:
def tensor_product(rep1: np.array, rep2: np.array) -> np.array:
    """Returns tensor product of two representations.
    Input:
        rep1: np.array [n, d1, d1] a representation of the group.
        rep2: np.array [n, d2, d2] another representation of the group.
    Output:
        Tensor product of rep1 and rep2. np.array [n, d1*d2, d1*d2], equal to the Kronecker product of rep1[i,:,:] and rep2[i,:,:] at group element i
    """
    pass  # your code here

In [ ]:
np.testing.assert_allclose(
    tensor_product(np.array([[[1.0]], [[-1.0]]]), np.array([[[1.0]], [[-1.0]]])),
    np.array([[[1.0]], [[1.0]]]),
)

---
## 7. `reduce_tensor_product(rep1, rep2, rep3)`

Compute a basis for all similarity transforms between the tensor product of `rep1` and `rep2`, and `rep3`. The output $Q$ satisfies $(\rho_1 \otimes \rho_2) Q = Q \rho_3$.

Clebsch–Gordan coefficients are a special case where `rep3` ranges over all irreps.

You may find `linalg.infer_change_of_basis` useful.

In [ ]:
def reduce_tensor_product(rep1: np.array, rep2: np.array, rep3: np.array) -> np.ndarray:
    """Returns the change of basis matrix that reduces the tensor product of rep1 and rep2 into rep3.
    Input:
        rep1: np.array [n, d1, d1]
        rep2: np.array [n, d2, d2]
        rep3: np.array [n, d3, d3]
    Output:
        Basis of the space of change of basis. mat = np.array [n_sol, d1, d2, d3]
            where each Q=mat[i].reshape(d1*d2,d3) satisfies tensor_product(rep1, rep2) @ Q = Q @ rep3,
            and such that the mat[i] matrices together form a basis for the subspace of all such Q.
    """
    pass  # your code here

In [ ]:
p3_irrep_sign = np.array([[[1.0]], [[-1.0]], [[-1.0]], [[-1.0]], [[1.0]], [[1.0]]])
p3_irrep_rot = np.array([
    [[1.0, 0.0], [0.0, 1.0]],
    [[-0.2916587, 0.95652245], [0.95652245, 0.2916587]],
    [[-0.6825434, -0.73084507], [-0.73084507, 0.6825434]],
    [[0.97420209, -0.22567739], [-0.22567739, -0.97420209]],
    [[-0.5, 0.8660254], [-0.8660254, -0.5]],
    [[-0.5, -0.8660254], [0.8660254, -0.5]],
])
c = reduce_tensor_product(p3_irrep_rot, p3_irrep_sign, p3_irrep_rot)
np.testing.assert_allclose(
    np.einsum('gab,gcd,sace->sgebd', p3_irrep_rot, p3_irrep_sign, c),
    np.einsum('sabc,gdc->sgdab', c, p3_irrep_rot),
)